In [ ]:
# %pip install -U pip
# %pip install -U pandas numpy scipy scikit-learn tqdm datasets sentencepiece
# %pip install -U torch transformers accelerate peft bitsandbytes

In [ ]:
from pathlib import Path
import json
import os
import random
import math
import re
from collections import Counter
from dataclasses import dataclass
from typing import Iterable, List

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

In [ ]:
PROJECT_DIR = Path.cwd()
RAW_DATA_DIR = PROJECT_DIR / 'data_raw' #input
NEWS_ONLY_DIR = PROJECT_DIR / 'news_only_data' #output
SFT_OUTPUT_DIR = PROJECT_DIR / 'sft_run_inline' #output

for path in [RAW_DATA_DIR, NEWS_ONLY_DIR, SFT_OUTPUT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

CONFIG = {
    'project_dir': str(PROJECT_DIR),
    'raw_data_dir': str(RAW_DATA_DIR),
    'news_only_dir': str(NEWS_ONLY_DIR),
    'sft_output_dir': str(SFT_OUTPUT_DIR),
    'seed': 42,
    'top_k_articles': 4,
    'top_k_sentences': 2,
    'max_sentence_chars': 280,
    'max_evidence_chars': 2400,
    'model_id': 'Qwen/Qwen2.5-1.5B-Instruct',
    'max_length': 896,
    'learning_rate': 1.5e-4,
    'epochs': 3,
    'batch_size': 8,
    'grad_accum': 4,
    'lora_r': 16,
    'lora_alpha': 32,
    'lora_dropout': 0.05,
    'weight_decay': 0.01,
    'warmup_ratio': 0.05,
    'calibration_size': 0.15,
    'calibration_method': 'platt',
    'use_class_weighting': True,
}
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))


In [ ]:
required = [
    'final_train_data.csv',
    'final_valid_data.csv',
    'final_test_data.csv',
]
missing = [name for name in required if not (RAW_DATA_DIR / name).exists()]
if missing:
    raise FileNotFoundError('Не найдены файлы в data_raw: ' + ', '.join(missing))

print('Исходные файлы найдены:')
for name in required:
    print('-', RAW_DATA_DIR / name)


In [ ]:
tqdm.pandas()

# пересывание промта под новый вид
LEAK_RE = re.compile(
    r"[ \t]*Please note that for the YES event, "
    r"Polymarket participants assign a probability of[^\n]*\n?",
    re.I,
)

# отбор нужных данных для подачи запроса
QUESTION_RE = re.compile(r"Question:\s*(.+?)\n", re.S)
DATE_RE = re.compile(r"Question close date:\s*(.+?)\n", re.S)
ARTICLE_RE = re.compile(
    r"Article \[(\d+)\]\n"
    r"Title:\s*(.*?)\n"
    r"Published date:\s*(.*?)\n"
    r"Author:\s*(.*?)\n"
    r"URL:\s*(.*?)\n"
    r"Text:\s*(.*?)(?=\nArticle \[\d+\]\n|\nReturn your final answer as a probability between 0 and 1\.?\s*$|$)",
    re.S,
)
TOKEN_RE = re.compile(r"[A-Za-z0-9']+")
SENT_SPLIT_RE = re.compile(r"(?<=[.!?])\s+")
STOPWORDS = {
    'the', 'a', 'an', 'and', 'or', 'to', 'of', 'in', 'on', 'for', 'with', 'at', 'by',
    'from', 'will', 'be', 'is', 'are', 'was', 'were', 'this', 'that', 'it', 'as', 'if',
    'about', 'into', 'than', 'between', 'their', 'they', 'them', 'his', 'her', 'its'
}

@dataclass
class Article:
    article_id: int
    title: str
    published_date: str
    author: str
    url: str
    text: str


def normalize_ws(text: str) -> str:
    return re.sub(r"\s+", ' ', str(text or '')).strip()


def tokenize(text: str) -> List[str]:
    return [
        token.lower()
        for token in TOKEN_RE.findall(text.lower())
        if token.lower() not in STOPWORDS and len(token) > 2
    ]


def parse_prompt(prompt: str):
    text = LEAK_RE.sub('', str(prompt or ''))
    q_match = QUESTION_RE.search(text)
    d_match = DATE_RE.search(text)
    question = normalize_ws(q_match.group(1)) if q_match else ''
    close_date = normalize_ws(d_match.group(1)) if d_match else ''
    articles = []
    for match in ARTICLE_RE.finditer(text):
        articles.append(
            Article(
                article_id=int(match.group(1)),
                title=normalize_ws(match.group(2)),
                published_date=normalize_ws(match.group(3)),
                author=normalize_ws(match.group(4)),
                url=normalize_ws(match.group(5)),
                text=normalize_ws(match.group(6)),
            )
        )
    return question, close_date, articles


def score_article(question_tokens: List[str], article: Article) -> float:
    title_tokens = tokenize(article.title)
    body_tokens = tokenize(article.text[:2500])
    overlap_title = sum((Counter(title_tokens) & Counter(question_tokens)).values())
    overlap_body = sum((Counter(body_tokens) & Counter(question_tokens)).values())
    recency_bonus = 0.0
    if article.published_date:
        year_match = re.match(r"(\d{4})-", article.published_date)
        if year_match:
            recency_bonus = max(0, int(year_match.group(1)) - 2020) * 0.25
    return overlap_title * 3.0 + overlap_body * 1.0 + recency_bonus


def score_sentence(question_tokens: List[str], sentence: str) -> float:
    sent_tokens = tokenize(sentence)
    overlap = sum((Counter(sent_tokens) & Counter(question_tokens)).values())
    has_number = bool(re.search(r"\d", sentence))
    return overlap + (0.5 if has_number else 0.0)


def choose_sentences(article: Article, question_tokens: List[str], top_k_sentences: int, max_sentence_chars: int):
    raw_sentences = [normalize_ws(s) for s in SENT_SPLIT_RE.split(article.text) if normalize_ws(s)]
    scored = sorted(raw_sentences, key=lambda s: score_sentence(question_tokens, s), reverse=True)
    selected = []
    for sent in scored:
        clipped = sent[:max_sentence_chars].strip()
        if len(clipped) < 30:
            continue
        selected.append(clipped)
        if len(selected) >= top_k_sentences:
            break
    return selected


def build_evidence(question: str, close_date: str, articles: List[Article], top_k_articles: int, top_k_sentences: int, max_sentence_chars: int, max_evidence_chars: int):
    q_tokens = tokenize(question)
    ranked_articles = sorted(articles, key=lambda art: score_article(q_tokens, art), reverse=True)[:top_k_articles]

    evidence_parts = [
        f'Question: {question}',
        f'Close date: {close_date}',
        '',
        'Evidence:',
    ]
    metadata = []
    for art in ranked_articles:
        picked = choose_sentences(art, q_tokens, top_k_sentences, max_sentence_chars)
        if not picked:
            continue
        part = [
            f'- Title: {art.title}',
            f'  Published: {art.published_date}',
        ]
        for sent in picked:
            part.append(f'  - {sent}')
        evidence_parts.append('\n'.join(part))
        metadata.append({
            'article_id': art.article_id,
            'title': art.title,
            'published_date': art.published_date,
            'url': art.url,
            'selected_sentences': picked,
        })

    evidence = '\n'.join(evidence_parts).strip()
    evidence = evidence[:max_evidence_chars].strip()
    return evidence, metadata


def convert_split(df: pd.DataFrame, split_name: str) -> pd.DataFrame:
    rows = []
    iterator = tqdm(df.iterrows(), total=len(df), desc=f'prepare {split_name}')
    for _, row in iterator:
        question, close_date, articles = parse_prompt(row['prompt'])
        evidence_text, selected_articles = build_evidence(
            question=question or str(row.get('question_text', '')),
            close_date=close_date or str(row.get('question_close_date', '')),
            articles=articles,
            top_k_articles=CONFIG['top_k_articles'],
            top_k_sentences=CONFIG['top_k_sentences'],
            max_sentence_chars=CONFIG['max_sentence_chars'],
            max_evidence_chars=CONFIG['max_evidence_chars'],
        )
        rows.append({
            'question_text': question or str(row.get('question_text', '')),
            'question_close_date': close_date or str(row.get('question_close_date', '')),
            'volume': row.get('volume', np.nan),
            'resolution': row.get('resolution', np.nan),
            'evidence_text': evidence_text,
            'selected_articles_json': json.dumps(selected_articles, ensure_ascii=False),
        })
    return pd.DataFrame(rows)


In [ ]:
train_raw = pd.read_csv(RAW_DATA_DIR / 'final_train_data.csv')
valid_raw = pd.read_csv(RAW_DATA_DIR / 'final_valid_data.csv')
test_raw = pd.read_csv(RAW_DATA_DIR / 'final_test_data.csv')

train_news = convert_split(train_raw, 'train')
valid_news = convert_split(valid_raw, 'valid')
test_news = convert_split(test_raw, 'test')

# новые датасеты с учетом промта и скрытия вероятностей полимаркета
train_news.to_csv(NEWS_ONLY_DIR / 'train_news_only.csv', index=False, encoding='utf-8-sig')
valid_news.to_csv(NEWS_ONLY_DIR / 'valid_news_only.csv', index=False, encoding='utf-8-sig')
test_news.to_csv(NEWS_ONLY_DIR / 'test_news_only.csv', index=False, encoding='utf-8-sig')

print('Saved:')
print('-', NEWS_ONLY_DIR / 'train_news_only.csv')
print('-', NEWS_ONLY_DIR / 'valid_news_only.csv')
print('-', NEWS_ONLY_DIR / 'test_news_only.csv')


In [ ]:
# перепроверка данных
print('train columns:', list(train_news.columns))
assert 'prediction_polymarket' not in train_news.columns
assert 'prediction_polymarket' not in valid_news.columns
assert 'prediction_polymarket' not in test_news.columns
print('OK: prediction_polymarket отсутствует во всех news-only split.')
train_news[['question_text', 'resolution', 'evidence_text']].head(3)


In [ ]:
PROMPT_TEMPLATE = '''Forecast the probability that the event resolves YES using only the news evidence.

Question: {question}
Close date: {close_date}

Evidence:
{evidence}
'''

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def sigmoid(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    return 1.0 / (1.0 + np.exp(-np.clip(x, -40, 40)))


def safe_brier(y_true: Iterable[float], probs: Iterable[float]) -> float:
    return float(brier_score_loss(np.asarray(list(y_true), dtype=float), np.asarray(list(probs), dtype=float)))


def safe_auc(y_true: Iterable[float], probs: Iterable[float]):
    y_true = np.asarray(list(y_true), dtype=float)
    probs = np.asarray(list(probs), dtype=float)
    if len(np.unique(y_true)) < 2:
        return None
    return float(roc_auc_score(y_true, probs))


def build_text(df: pd.DataFrame) -> List[str]:
    return [
        PROMPT_TEMPLATE.format(
            question=str(q or ''),
            close_date=str(d or ''),
            evidence=str(e or ''),
        )
        for q, d, e in zip(df['question_text'], df['question_close_date'], df['evidence_text'])
    ]


def prepare_dataset(df: pd.DataFrame, tokenizer, max_length: int) -> Dataset:
    texts = build_text(df)
    labels = df['resolution'].astype(float).tolist()
    dataset = Dataset.from_dict({'text': texts, 'labels': labels})

    def tokenize_batch(batch):
        out = tokenizer(
            batch['text'],
            truncation=True,
            max_length=max_length,
            padding=False,
        )
        out['labels'] = batch['labels']
        return out

    return dataset.map(tokenize_batch, batched=True, remove_columns=['text'], desc='tokenize')


class WeightedBCETrainer(Trainer):
    def __init__(self, *args, pos_weight: float = 1.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.pos_weight = float(max(pos_weight, 1e-6))

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        inputs = inputs.copy()
        labels = inputs.pop('labels').float()
        outputs = model(**inputs)
        logits = outputs.logits.reshape(-1)
        labels = labels.to(logits.device).reshape(-1)
        pos_weight = torch.tensor(self.pos_weight, device=logits.device, dtype=logits.dtype)
        loss = torch.nn.functional.binary_cross_entropy_with_logits(logits, labels, pos_weight=pos_weight)
        return (loss, outputs) if return_outputs else loss


class ProbabilityCalibrator:
    def __init__(self, method: str):
        self.method = method
        self.model = None

    def fit(self, logits: np.ndarray, y: np.ndarray) -> None:
        logits = np.asarray(logits, dtype=np.float64).reshape(-1)
        y = np.asarray(y, dtype=np.float64).reshape(-1)
        if self.method == 'none':
            return
        if self.method == 'platt':
            clf = LogisticRegression(max_iter=1000, solver='lbfgs')
            clf.fit(logits.reshape(-1, 1), y)
            self.model = clf
            return
        if self.method == 'isotonic':
            ir = IsotonicRegression(out_of_bounds='clip')
            ir.fit(sigmoid(logits), y)
            self.model = ir
            return
        raise ValueError(f'Unsupported calibration method: {self.method}')

    def transform(self, logits: np.ndarray) -> np.ndarray:
        logits = np.asarray(logits, dtype=np.float64).reshape(-1)
        raw_probs = sigmoid(logits)
        if self.method == 'none' or self.model is None:
            return np.clip(raw_probs, 1e-4, 1 - 1e-4)
        if self.method == 'platt':
            probs = self.model.predict_proba(logits.reshape(-1, 1))[:, 1]
            return np.clip(probs, 1e-4, 1 - 1e-4)
        if self.method == 'isotonic':
            probs = self.model.transform(raw_probs)
            return np.clip(probs, 1e-4, 1 - 1e-4)
        raise ValueError(f'Unsupported calibration method: {self.method}')


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    logits = np.asarray(logits).reshape(-1)
    probs = np.clip(sigmoid(logits), 1e-4, 1 - 1e-4)
    auc = safe_auc(labels, probs)
    return {
        'brier_raw': safe_brier(labels, probs),
        'auc_raw': float('nan') if auc is None else auc,
    }


def split_train_and_calibration(train_df: pd.DataFrame, calibration_size: float, seed: int):
    if calibration_size <= 0.0 or len(train_df) < 100:
        return train_df.reset_index(drop=True), None
    calib_size = min(max(calibration_size, 0.05), 0.4)
    if train_df['resolution'].nunique() < 2:
        return train_df.reset_index(drop=True), None
    fit_df, calib_df = train_test_split(
        train_df,
        test_size=calib_size,
        random_state=seed,
        stratify=train_df['resolution'],
    )
    return fit_df.reset_index(drop=True), calib_df.reset_index(drop=True)


def predict_logits(trainer: Trainer, dataset: Dataset) -> np.ndarray:
    return trainer.predict(dataset).predictions.reshape(-1)


def collect_split_metrics(df: pd.DataFrame, probs_raw: np.ndarray, probs_cal: np.ndarray) -> dict:
    y = df['resolution'].astype(float).to_numpy()
    return {
        'rows': int(len(df)),
        'positive_rate': float(y.mean()),
        'brier_raw': safe_brier(y, probs_raw),
        'auc_raw': safe_auc(y, probs_raw),
        'brier_calibrated': safe_brier(y, probs_cal),
        'auc_calibrated': safe_auc(y, probs_cal),
        'prediction_mean_raw': float(np.mean(probs_raw)),
        'prediction_mean_calibrated': float(np.mean(probs_cal)),
        'prediction_std_raw': float(np.std(probs_raw)),
        'prediction_std_calibrated': float(np.std(probs_cal)),
        'unique_probs_raw': int(pd.Series(np.round(probs_raw, 6)).nunique()),
        'unique_probs_calibrated': int(pd.Series(np.round(probs_cal, 6)).nunique()),
    }


set_seed(CONFIG['seed'])
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
    print('vram_gb:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))


## Обучение

In [ ]:
train_df_full = pd.read_csv(NEWS_ONLY_DIR / 'train_news_only.csv')
valid_df = pd.read_csv(NEWS_ONLY_DIR / 'valid_news_only.csv')
test_df = pd.read_csv(NEWS_ONLY_DIR / 'test_news_only.csv')
train_df, calib_df = split_train_and_calibration(train_df_full, CONFIG['calibration_size'], CONFIG['seed'])

tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_id'], trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForSequenceClassification.from_pretrained(
    CONFIG['model_id'],
    num_labels=1,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model.config.pad_token_id = tokenizer.pad_token_id
if hasattr(model, 'gradient_checkpointing_enable'):
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
if hasattr(model, 'enable_input_require_grads'):
    model.enable_input_require_grads()
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=CONFIG['lora_r'],
    lora_alpha=CONFIG['lora_alpha'],
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=CONFIG['lora_dropout'],
    bias='none',
    task_type='SEQ_CLS',
)
model = get_peft_model(model, lora_config)

train_ds = prepare_dataset(train_df, tokenizer, CONFIG['max_length'])
eval_df = calib_df if calib_df is not None else valid_df
eval_ds = prepare_dataset(eval_df, tokenizer, CONFIG['max_length'])
valid_ds = prepare_dataset(valid_df, tokenizer, CONFIG['max_length'])
test_ds = prepare_dataset(test_df, tokenizer, CONFIG['max_length'])
full_train_ds = prepare_dataset(train_df_full, tokenizer, CONFIG['max_length'])
calib_ds = prepare_dataset(calib_df, tokenizer, CONFIG['max_length']) if calib_df is not None else None
collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)

y_train = train_df['resolution'].astype(float).to_numpy()
pos_rate = float(y_train.mean())
neg_rate = max(1.0 - pos_rate, 1e-6)
pos_weight = max(neg_rate / max(pos_rate, 1e-6), 1.0) if CONFIG['use_class_weighting'] else 1.0

training_args = TrainingArguments(
    output_dir=str(SFT_OUTPUT_DIR / 'trainer_artifacts'),
    learning_rate=CONFIG['learning_rate'],
    num_train_epochs=CONFIG['epochs'],
    per_device_train_batch_size=CONFIG['batch_size'],
    per_device_eval_batch_size=CONFIG['batch_size'],
    gradient_accumulation_steps=CONFIG['grad_accum'],
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_steps=10,
    bf16=False,
    fp16=True,
    report_to=[],
    load_best_model_at_end=True,
    metric_for_best_model='brier_raw',
    greater_is_better=False,
    remove_unused_columns=False,
    seed=CONFIG['seed'],
    weight_decay=CONFIG['weight_decay'],
    warmup_ratio=CONFIG['warmup_ratio'],
    disable_tqdm=False,
)

trainer = WeightedBCETrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
    pos_weight=pos_weight,
)

trainer.train()


In [ ]:
# результат
full_train_logits = predict_logits(trainer, full_train_ds)
valid_logits = predict_logits(trainer, valid_ds)
test_logits = predict_logits(trainer, test_ds)

calibration_logits = full_train_logits
calibration_targets = train_df_full['resolution'].astype(float).to_numpy()
calibration_source = 'full_train'
if calib_df is not None and calib_ds is not None and len(calib_df) > 0:
    calibration_logits = predict_logits(trainer, calib_ds)
    calibration_targets = calib_df['resolution'].astype(float).to_numpy()
    calibration_source = 'heldout_calibration_split'

calibrator = ProbabilityCalibrator(CONFIG['calibration_method'])
calibrator.fit(calibration_logits, calibration_targets)

full_train_probs_raw = np.clip(sigmoid(full_train_logits), 1e-4, 1 - 1e-4)
valid_probs_raw = np.clip(sigmoid(valid_logits), 1e-4, 1 - 1e-4)
test_probs_raw = np.clip(sigmoid(test_logits), 1e-4, 1 - 1e-4)

full_train_probs_cal = calibrator.transform(full_train_logits)
valid_probs_cal = calibrator.transform(valid_logits)
test_probs_cal = calibrator.transform(test_logits)

for split_name, df, raw_preds, cal_preds in [
    ('train', train_df_full.copy(), full_train_probs_raw, full_train_probs_cal),
    ('valid', valid_df.copy(), valid_probs_raw, valid_probs_cal),
    ('test', test_df.copy(), test_probs_raw, test_probs_cal),
]:
    df['prediction_sft_news_only_raw'] = raw_preds
    df['prediction_sft_news_only'] = cal_preds
    df.to_csv(
        SFT_OUTPUT_DIR / f'{split_name}_news_only_sft_predictions.csv',
        index=False,
        encoding='utf-8-sig',
    )

summary = {
    'model_id': CONFIG['model_id'],
    'uses_polymarket_probability': False,
    'calibration_method': CONFIG['calibration_method'],
    'calibration_source': calibration_source,
    'train_rows_full': int(len(train_df_full)),
    'train_rows_fit': int(len(train_df)),
    'calibration_rows': int(0 if calib_df is None else len(calib_df)),
    'valid_rows': int(len(valid_df)),
    'test_rows': int(len(test_df)),
    'max_length': int(CONFIG['max_length']),
    'epochs': int(CONFIG['epochs']),
    'learning_rate': float(CONFIG['learning_rate']),
    'pos_weight': float(pos_weight),
    'class_weighting_enabled': bool(CONFIG['use_class_weighting']),
    'train_metrics': collect_split_metrics(train_df_full, full_train_probs_raw, full_train_probs_cal),
    'valid_metrics': collect_split_metrics(valid_df, valid_probs_raw, valid_probs_cal),
    'test_metrics': collect_split_metrics(test_df, test_probs_raw, test_probs_cal),
}
if calib_df is not None and len(calib_df) > 0:
    calib_probs_raw = np.clip(sigmoid(calibration_logits), 1e-4, 1 - 1e-4)
    calib_probs_cal = calibrator.transform(calibration_logits)
    summary['calibration_metrics'] = collect_split_metrics(calib_df, calib_probs_raw, calib_probs_cal)

(SFT_OUTPUT_DIR / 'sft_summary.json').write_text(
    json.dumps(summary, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
trainer.model.save_pretrained(SFT_OUTPUT_DIR / 'sft_adapter')
tokenizer.save_pretrained(SFT_OUTPUT_DIR / 'sft_adapter')
print(json.dumps(summary, ensure_ascii=False, indent=2))


In [ ]:
summary = json.loads((SFT_OUTPUT_DIR / 'sft_summary.json').read_text(encoding='utf-8'))
print(json.dumps(summary, ensure_ascii=False, indent=2))

valid_pred = pd.read_csv(SFT_OUTPUT_DIR / 'valid_news_only_sft_predictions.csv')
print('columns:', list(valid_pred.columns))
valid_pred[[
    'question_text',
    'resolution',
    'prediction_sft_news_only_raw',
    'prediction_sft_news_only',
]].head(10)
